In [2]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpImage
from PIL import Image
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from fastapi import FastAPI, File, UploadFile

In [3]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.cnn = nn.Sequential(

        nn.Conv2d(3, 32, kernel_size=(3,3), padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2)),

        nn.Conv2d(32, 64, kernel_size=(3,3), padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2)),

        nn.Conv2d(64, 128, kernel_size=(3,3), padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2)),
    )

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(16*16*128, 256),
        nn.ReLU(),

        nn.Linear(256, 1)

    )


  def forward(self, x):
    x = self.cnn(x)
    x = self.fc(x)

    return x

In [4]:
model = CNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

In [5]:
model.load_state_dict(torch.load('face_mask_best.pt', map_location='cpu'))

<All keys matched successfully>

In [6]:
def pred(image_path):
    model.eval()

    image = Image.open(image_path)
    image = image.resize((128, 128))
    image = image.convert('RGB')

    image = np.array(image) / 255.0

    image_tensor = torch.tensor(image, dtype=torch.float32)
    image_tensor = image_tensor.permute(2, 0, 1)  # (C,H,W)
    image_tensor = image_tensor.unsqueeze(0)      # (1,C,H,W)

    with torch.no_grad():
        output = model(image_tensor)
        prob = torch.sigmoid(output)
        pred = (prob > 0.5).float()

    return pred, prob.item()

In [11]:
def predict_image(image:Image.Image):
    model.eval()
    image = Image.open(image)
    image = image.resize((128, 128))
    image = image.convert('RGB')

    image = np.array(image) / 255.0

    image_tensor = torch.tensor(image, dtype=torch.float32)
    image_tensor = image_tensor.permute(2, 0, 1)  # (C,H,W)
    image_tensor = image_tensor.unsqueeze(0)      # (1,C,H,W)

    with torch.no_grad():
        output = model(image_tensor)
        prob = torch.sigmoid(output)
        pred = (prob > 0.5).float()

    return pred, prob.item()

In [8]:
app = FastAPI()

In [9]:
@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    
    prediction = predict_image(image)
    
    return {
        "prediction": int(prediction)
    }